# Hidden Emotion Detection — Stage 3 Retrain (Fixed Divergence Loss)

**What this notebook does:**
- Loads Stage 1 (HuBERT fine-tuned) and Stage 2 (SSL adapters) checkpoints from Drive
- Rebuilds all model classes identically to the original notebook
- Retrains **only Stage 3** with the corrected divergence loss (negative KL + raw cosine penalty)
- Evaluates on the Session 5 test set with full hidden emotion analysis
- Re-runs Cell 12 validation against annotator disagreement entropy

**Key fix:** `loss = CE_exp + CE_hid + λ_orth * cosine_sim - λ_div * KL`
(was `+ KL` and `+ cosine_sim.abs()` — both actively collapsed the heads)


## Cell 1 — Install dependencies

In [ ]:
!pip install transformers datasets torchaudio scikit-learn pandas scipy kaggle -q
print("Install complete")

## Cell 2 — Mount Google Drive & Kaggle API

In [ ]:
from google.colab import drive, files
import os

drive.mount('/content/drive')

print('Upload your kaggle.json file:')
uploaded = files.upload()
if uploaded:
    fname = list(uploaded.keys())[0]
    os.makedirs('/root/.kaggle', exist_ok=True)
    os.rename(fname, '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    print('Kaggle API ready')
else:
    print('No file uploaded.')

## Cell 3 — Download IEMOCAP & CREMA-D

In [ ]:
import kagglehub

# IEMOCAP
print('Downloading IEMOCAP...')
iemocap_path = kagglehub.dataset_download('dejolilandry/iemocapfullrelease')
print(f'IEMOCAP path: {iemocap_path}')

# CREMA-D
print('Downloading CREMA-D...')
crema_path = kagglehub.dataset_download('ejlok1/cremad')

# Auto-detect CREMA-D wav directory
crema_data_root = None
candidate = os.path.join(crema_path, 'AudioWAV')
if os.path.isdir(candidate) and any(f.endswith('.wav') for f in os.listdir(candidate)):
    crema_data_root = candidate
else:
    for root, dirs, fs in os.walk(crema_path):
        if any(f.endswith('.wav') for f in fs):
            crema_data_root = root
            break

if crema_data_root is None:
    raise RuntimeError(f'No WAV files found under {crema_path}')

n_wavs = len([f for f in os.listdir(crema_data_root) if f.endswith('.wav')])
print(f'CREMA-D audio root : {crema_data_root}')
print(f'Total WAV files    : {n_wavs}')

# IEMOCAP root — find directory containing Session1..Session5
IEMOCAP_ROOT = None
for root, dirs, fs in os.walk(iemocap_path):
    if 'Session1' in dirs or 'Session2' in dirs:
        IEMOCAP_ROOT = root
        break
if IEMOCAP_ROOT is None:
    IEMOCAP_ROOT = iemocap_path
print(f'IEMOCAP root: {IEMOCAP_ROOT}')

## Cell 4 — Imports & Config

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import torchaudio
import torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import HubertForSequenceClassification, Wav2Vec2FeatureExtractor
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from collections import Counter

DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

SAMPLE_RATE = 16000
MAX_LENGTH  = 6 * SAMPLE_RATE
BATCH_SIZE  = 16
EPOCHS_S3   = 10       # extra epochs since we're fixing a collapsed model
GRAD_ACCUM  = 2

EMOTIONS  = ['angry', 'happy', 'neutral', 'sad']
label2id  = {e: i for i, e in enumerate(EMOTIONS)}
id2label  = {i: e for i, e in enumerate(EMOTIONS)}

# ── Divergence loss weights (THE KEY FIX) ──────────────────────────────────
# λ_div  : weight on -KL term (pushes heads apart). Start 0.3, bump if needed.
# λ_orth : weight on cosine_sim penalty (no .abs() — rewards dissimilarity)
λ_div  = 0.3
λ_orth = 0.1

print(f'Classes ({len(EMOTIONS)}): {EMOTIONS}')
print(f'λ_div={λ_div}  λ_orth={λ_orth}')
print(f'Loss = CE_exp + CE_hid + {λ_orth}*cosine_sim - {λ_div}*KL')
print('       (negative KL = push heads APART)')

## Cell 5 — Model Definitions (identical to original notebook)

In [ ]:
class AdapterLayer(nn.Module):
    def __init__(self, hidden_size=768, bottleneck=64):
        super().__init__()
        self.down = nn.Linear(hidden_size, bottleneck)
        self.act  = nn.GELU()
        self.up   = nn.Linear(bottleneck, hidden_size)
        self.norm = nn.LayerNorm(hidden_size)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return self.norm(x + self.up(self.act(self.down(x))))


class HuBERTWithAdapters(nn.Module):
    def __init__(self, hubert_model):
        super().__init__()
        self.hubert     = hubert_model.hubert
        self.num_layers = len(self.hubert.encoder.layers)
        self.adapters   = nn.ModuleList([AdapterLayer() for _ in range(self.num_layers)])
        for param in self.hubert.parameters():
            param.requires_grad = False

    def forward(self, input_values):
        hidden = self.hubert(input_values=input_values,
                             output_hidden_states=True).last_hidden_state
        for adapter in self.adapters:
            hidden = adapter(hidden)
        return hidden


class DualPathAttention(nn.Module):
    def __init__(self, hidden_size=768, window=4):
        super().__init__()
        self.window   = window
        self.scale    = hidden_size ** 0.5
        self.local_q  = nn.Linear(hidden_size, hidden_size)
        self.local_k  = nn.Linear(hidden_size, hidden_size)
        self.local_v  = nn.Linear(hidden_size, hidden_size)
        self.global_q = nn.Parameter(torch.randn(1, 1, hidden_size))
        self.global_k = nn.Linear(hidden_size, hidden_size)
        self.global_v = nn.Linear(hidden_size, hidden_size)
        self.norm_local  = nn.LayerNorm(hidden_size)
        self.norm_global = nn.LayerNorm(hidden_size)

    def forward(self, x):
        B, T, H = x.shape
        W = self.window
        Q = self.local_q(x)
        K = self.local_k(x)
        V = self.local_v(x)
        K_pad = F.pad(K.transpose(1, 2), (W, W)).transpose(1, 2)
        V_pad = F.pad(V.transpose(1, 2), (W, W)).transpose(1, 2)
        win_size = 2 * W + 1
        K_win = torch.stack([K_pad[:, t:t+T, :] for t in range(win_size)], dim=2)
        V_win = torch.stack([V_pad[:, t:t+T, :] for t in range(win_size)], dim=2)
        w_local      = torch.softmax(torch.einsum('bth,btwh->btw', Q, K_win) / self.scale, dim=-1)
        local_out    = torch.einsum('btw,btwh->bth', w_local, V_win)
        local_weights = w_local.mean(dim=-1)
        local_pooled = local_out.mean(dim=1)
        gq           = self.global_q.expand(B, -1, -1)
        gk           = self.global_k(x)
        gv           = self.global_v(x)
        g_scores     = torch.bmm(gq, gk.transpose(1, 2)) / self.scale
        g_weights    = torch.softmax(g_scores, dim=-1)
        global_out   = torch.bmm(g_weights, gv).squeeze(1)
        global_weights = g_weights.squeeze(1)
        return self.norm_local(local_pooled), self.norm_global(global_out), local_weights, global_weights


class HiddenEmotionModel(nn.Module):
    def __init__(self, adapter_model, num_labels=4, hidden_size=768):
        super().__init__()
        self.encoder       = adapter_model
        self.dual_attn     = DualPathAttention(hidden_size)
        self.explicit_head = nn.Sequential(
            nn.Linear(hidden_size, 256), nn.GELU(), nn.Dropout(0.1), nn.Linear(256, num_labels)
        )
        self.hidden_head = nn.Sequential(
            nn.Linear(hidden_size, 256), nn.GELU(), nn.Dropout(0.1), nn.Linear(256, num_labels)
        )

    def forward(self, input_values):
        hidden = self.encoder(input_values)
        local_rep, global_rep, lw, gw = self.dual_attn(hidden)
        explicit_logits = self.explicit_head(global_rep)
        hidden_logits   = self.hidden_head(local_rep)
        eps       = 1e-8
        exp_probs = F.softmax(explicit_logits, dim=-1)
        hid_probs = F.softmax(hidden_logits,   dim=-1)
        kl_div = (exp_probs * ((exp_probs + eps) / (hid_probs + eps)).log()).sum(dim=-1)
        return explicit_logits, hidden_logits, kl_div


def save_checkpoint(obj, name):
    state = obj.state_dict() if hasattr(obj, 'state_dict') else obj
    for p in [f'/content/{name}.pt', f'/content/drive/MyDrive/{name}.pt']:
        torch.save(state, p)
    print(f'Saved {name}')


def load_checkpoint(obj, name):
    local = f'/content/{name}.pt'
    drv   = f'/content/drive/MyDrive/{name}.pt'
    path  = local if os.path.exists(local) else drv
    if not os.path.exists(path):
        raise FileNotFoundError(f'Checkpoint not found: {name}.pt (checked /content and Drive)')
    obj.load_state_dict(torch.load(path, map_location=DEVICE))
    print(f'Loaded {name} from {path}')
    return obj

print('Model classes defined')

## Cell 6 — Dataset Classes & IEMOCAP Loader

In [ ]:
def load_iemocap(root_dir):
    label_map = {
        'hap': 'happy', 'exc': 'happy',
        'sad': 'sad',   'ang': 'angry',
        'neu': 'neutral'
    }
    records = []
    for session in range(1, 6):
        eval_dir = os.path.join(root_dir, f'Session{session}', 'dialog', 'EmoEvaluation')
        wav_dir  = os.path.join(root_dir, f'Session{session}', 'sentences', 'wav')
        if not os.path.exists(eval_dir):
            print(f'Warning: Session{session} not found, skipping')
            continue
        for fname in os.listdir(eval_dir):
            if not fname.endswith('.txt'):
                continue
            with open(os.path.join(eval_dir, fname)) as f:
                for line in f:
                    if not line.startswith('['):
                        continue
                    parts = line.strip().split('\t')
                    if len(parts) < 3:
                        continue
                    utt_id  = parts[1].strip()
                    emotion = parts[2].strip()
                    if emotion not in label_map:
                        continue
                    dialog   = '_'.join(utt_id.split('_')[:-1])
                    wav_path = os.path.join(wav_dir, dialog, utt_id + '.wav')
                    if not os.path.exists(wav_path):
                        continue
                    records.append({'wav_path': wav_path,
                                    'emotion': label_map[emotion],
                                    'session': session})
    df = pd.DataFrame(records)
    print(f'Loaded {len(df)} IEMOCAP utterances')
    print(df['emotion'].value_counts())
    return df


class IEMOCAPDataset(Dataset):
    def __init__(self, df, processor):
        self.df        = df.reset_index(drop=True)
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        wav, sr = torchaudio.load(row['wav_path'])
        if sr != SAMPLE_RATE:
            wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)
        wav = wav.mean(dim=0)[:MAX_LENGTH]
        inp = self.processor(
            wav.numpy(), sampling_rate=SAMPLE_RATE,
            return_tensors='pt', padding='max_length',
            max_length=MAX_LENGTH, truncation=True
        )
        return {
            'input_values': inp['input_values'].squeeze(0),
            'label':        torch.tensor(label2id[row['emotion']], dtype=torch.long)
        }


def make_balanced_sampler(dataset):
    labels  = [dataset[i]['label'].item() for i in range(len(dataset))]
    counts  = Counter(labels)
    weights = [1.0 / counts[l] for l in labels]
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

print('Dataset classes defined')

## Cell 7 — Load IEMOCAP & Build Splits

In [ ]:
from transformers import Wav2Vec2FeatureExtractor

processor = Wav2Vec2FeatureExtractor.from_pretrained('facebook/hubert-base-ls960')

df_all = load_iemocap(IEMOCAP_ROOT)

# Session 5 = test set (matches original notebook)
test_df  = df_all[df_all['session'] == 5].reset_index(drop=True)
train_df = df_all[df_all['session'] != 5].reset_index(drop=True)

# Val = 15% of train sessions, stratified
train_df_excl, val_df = train_test_split(
    train_df, test_size=0.15, stratify=train_df['emotion'], random_state=42
)
train_df_excl = train_df_excl.reset_index(drop=True)
val_df        = val_df.reset_index(drop=True)

print(f'Train: {len(train_df_excl)} | Val: {len(val_df)} | Test: {len(test_df)}')

val_loader  = DataLoader(IEMOCAPDataset(val_df,  processor),
                         batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(IEMOCAPDataset(test_df, processor),
                         batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print('Loaders ready')

## Cell 8 — Load Stage 1 & Stage 2 Checkpoints

In [ ]:
# Load base HuBERT model (same architecture as Stage 1)
model = HubertForSequenceClassification.from_pretrained(
    'facebook/hubert-base-ls960',
    num_labels=len(EMOTIONS),
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True
).to(DEVICE)

# Load Stage 1 fine-tuned weights
load_checkpoint(model, 'stage1_best')

# Wrap with adapters and load Stage 2 adapter weights
adapter_model = HuBERTWithAdapters(model).to(DEVICE)
load_checkpoint(adapter_model.adapters, 'stage2_adapters')

# Freeze HuBERT backbone; keep adapters trainable
for param in adapter_model.hubert.parameters():
    param.requires_grad = False
for param in adapter_model.adapters.parameters():
    param.requires_grad = True

# Build full model
full_model = HiddenEmotionModel(adapter_model, num_labels=len(EMOTIONS)).to(DEVICE)

trainable_params = sum(p.numel() for p in full_model.parameters() if p.requires_grad)
print(f'Trainable parameters: {trainable_params:,}')
print('Checkpoints loaded — ready for Stage 3 retrain')

## Cell 9 — Stage 3 Retrain (Fixed Divergence Loss)

**The two fixes:**
1. `- λ_div * kl_div.mean()` — **negative** KL pushes heads apart
2. `cosine_sim` without `.abs()` — rewards dissimilarity, penalises similarity


In [ ]:
print('\n=== Stage 3 (FIXED): Dual-path Attention + Corrected Divergence Loss ===')
print(f'Loss = CE_exp + CE_hid + {λ_orth}*cosine_sim - {λ_div}*KL')
print('Watch head_disagree — should climb from ~0.01 toward 0.20+ over epochs\n')

ce_loss  = nn.CrossEntropyLoss(label_smoothing=0.05)
best_val = 0.0

# Balanced train loader — IEMOCAP only, no CREMA-D
train_iemocap_only = IEMOCAPDataset(train_df_excl, processor)
s3_sampler         = make_balanced_sampler(train_iemocap_only)
s3_train_loader    = DataLoader(
    train_iemocap_only, batch_size=BATCH_SIZE,
    sampler=s3_sampler, num_workers=2, pin_memory=True
)

# Two param groups: adapters at small LR, new heads at larger LR
trainable = [
    {'params': list(full_model.encoder.adapters.parameters()), 'lr': 1e-5},
    {'params': (
        list(full_model.dual_attn.parameters()) +
        list(full_model.explicit_head.parameters()) +
        list(full_model.hidden_head.parameters())
    ), 'lr': 1e-4},
]
optimizer_s3 = torch.optim.AdamW(trainable, weight_decay=0.01)
scheduler_s3 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_s3, T_max=EPOCHS_S3)

current_λ_div = λ_div   # mutable — auto-bumped if heads don't separate

for epoch in range(EPOCHS_S3):
    full_model.train()
    total_loss          = 0.0
    correct             = 0
    total               = 0
    head_disagree_count = 0
    kl_running          = 0.0
    cosine_running      = 0.0

    optimizer_s3.zero_grad()

    for step, batch in enumerate(s3_train_loader):
        iv     = batch['input_values'].to(DEVICE)
        labels = batch['label'].to(DEVICE)

        hidden                        = full_model.encoder(iv)
        local_rep, global_rep, lw, gw = full_model.dual_attn(hidden)

        exp_logits = full_model.explicit_head(global_rep)
        hid_logits = full_model.hidden_head(local_rep)

        # ── KL over attention weight distributions ────────────────────────
        eps     = 1e-8
        lw_norm = (lw + eps) / (lw + eps).sum(dim=-1, keepdim=True)
        gw_norm = (gw + eps) / (gw + eps).sum(dim=-1, keepdim=True)
        kl_div  = (lw_norm * (lw_norm / gw_norm).log()).sum(dim=-1)

        loss_exp = ce_loss(exp_logits, labels)
        loss_hid = ce_loss(hid_logits, labels)

        # ── Orthogonality: raw cosine_sim (NO .abs()) ─────────────────────
        # Positive → similar (bad) → adds to loss
        # Negative → dissimilar (good) → subtracts from loss
        cosine_sim = F.cosine_similarity(global_rep, local_rep, dim=-1).mean()
        loss_orth  = cosine_sim * λ_orth

        # ── FIXED: NEGATIVE KL — penalises head agreement ─────────────────
        loss = loss_exp + loss_hid + loss_orth - current_λ_div * kl_div.mean()

        (loss / GRAD_ACCUM).backward()

        total_loss          += loss.item()
        correct             += (exp_logits.argmax(-1) == labels).sum().item()
        total               += labels.size(0)
        head_disagree_count += (exp_logits.argmax(-1) != hid_logits.argmax(-1)).sum().item()
        kl_running          += kl_div.mean().item()
        cosine_running      += cosine_sim.item()

        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(s3_train_loader):
            torch.nn.utils.clip_grad_norm_(full_model.parameters(), 1.0)
            optimizer_s3.step()
            optimizer_s3.zero_grad()

    scheduler_s3.step()

    # ── Validation ────────────────────────────────────────────────────────
    full_model.eval()
    vp, vt = [], []
    with torch.no_grad():
        for batch in val_loader:
            iv     = batch['input_values'].to(DEVICE)
            labels = batch['label'].to(DEVICE)
            exp_logits, _, _ = full_model(iv)
            vp.extend(exp_logits.argmax(-1).cpu().numpy())
            vt.extend(labels.cpu().numpy())

    val_acc       = accuracy_score(vt, vp)
    head_disagree = head_disagree_count / total
    avg_kl        = kl_running / len(s3_train_loader)
    avg_cosine    = cosine_running / len(s3_train_loader)

    is_best = val_acc > best_val
    if is_best:
        best_val = val_acc
        save_checkpoint(full_model, 'stage3_best_fixed')

    print(f'Epoch {epoch+1:2d}/{EPOCHS_S3} | '
          f'Loss: {total_loss/len(s3_train_loader):7.4f} | '
          f'Train: {correct/total:.3f} | '
          f'Val: {val_acc:.3f} | '
          f'HeadDisagree: {head_disagree:.3f} | '
          f'KL: {avg_kl:.4f} | '
          f'CosSim: {avg_cosine:.4f}'
          f'{"  <- best" if is_best else ""}')

    # ── Auto-adjust λ if heads not separating ─────────────────────────────
    if epoch == 1 and head_disagree < 0.05:
        current_λ_div = min(current_λ_div * 2, 1.0)
        print(f'  ⚠ Head disagreement still low — bumping λ_div to {current_λ_div:.2f}')
    if epoch == 3 and head_disagree < 0.10:
        current_λ_div = min(current_λ_div * 1.5, 2.0)
        print(f'  ⚠ Still low at epoch 4 — bumping λ_div to {current_λ_div:.2f}')

print(f'\nBest val acc: {best_val:.3f}')
print('Checkpoint saved as stage3_best_fixed')

## Cell 10 — Final Evaluation on Session 5 Test Set

In [ ]:
print('\n=== Final Evaluation (Session 5 Test Set) ===')
load_checkpoint(full_model, 'stage3_best_fixed')
full_model.eval()

all_exp, all_hid, all_kl, all_true = [], [], [], []

with torch.no_grad():
    for batch in test_loader:
        iv     = batch['input_values'].to(DEVICE)
        labels = batch['label'].to(DEVICE)

        exp_logits, hid_logits, kl = full_model(iv)

        all_exp.extend(exp_logits.argmax(-1).cpu().numpy())
        all_hid.extend(hid_logits.argmax(-1).cpu().numpy())
        all_kl.extend(kl.cpu().numpy())
        all_true.extend(labels.cpu().numpy())

all_exp  = np.array(all_exp)
all_hid  = np.array(all_hid)
all_kl   = np.array(all_kl)
all_true = np.array(all_true)

# ── Classification report ─────────────────────────────────────────────────
print(classification_report(all_true, all_exp, target_names=EMOTIONS, digits=2))

# ── Hidden emotion summary ────────────────────────────────────────────────
kl_threshold  = all_kl.mean() + all_kl.std()
flagged_mask  = all_kl > kl_threshold
head_disagree = (all_exp != all_hid)

print('=== Hidden Emotion Analysis ===')
print(f'  Mean KL divergence       : {all_kl.mean():.4f}')
print(f'  Std  KL divergence       : {all_kl.std():.4f}')
print(f'  Detection threshold      : {kl_threshold:.4f}')
print(f'  Likely suppressed        : {flagged_mask.sum()}/{len(flagged_mask)} '
      f'({100*flagged_mask.mean():.1f}%)')
print(f'  Head disagreement overall: {head_disagree.mean():.3f}')
print(f'  Head disagree (flagged)  : {head_disagree[flagged_mask].mean():.3f}')
print(f'  Head disagree (unflagged): {head_disagree[~flagged_mask].mean():.3f}')

print('\n=== Suppression Patterns (expressed → hidden, flagged utterances) ===')
patterns = Counter(zip(
    [id2label[e] for e in all_exp[flagged_mask]],
    [id2label[h] for h in all_hid[flagged_mask]]
))
n_flagged = flagged_mask.sum()
for (exp_e, hid_e), count in patterns.most_common(10):
    if exp_e != hid_e:
        print(f'  {exp_e:12s} -> {hid_e:12s} : {count:4d} ({100*count/n_flagged:.1f}%)')

## Cell 11 — Annotator Disagreement Entropy Validation (Cell 12)

Checks whether the KL-based suppression detector correlates with human annotator disagreement.
This is the publishable validation — run **after** the model shows healthy head disagreement (>15%).


In [ ]:
from scipy.stats import entropy as scipy_entropy, mannwhitneyu
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

VOTE_MAP = {
    'happiness': 'happy', 'excited': 'happy',
    'sadness': 'sad',     'anger': 'angry',
    'neutral': 'neutral',
    'frustrated': None, 'frustration': None,
    'fear': None, 'disgust': None, 'surprise': None, 'other': None,
}
MAJORITY_MAP = {
    'hap': 'happy', 'exc': 'happy',
    'sad': 'sad',   'ang': 'angry',
    'neu': 'neutral'
}


def load_iemocap_annotator_votes(root_dir, sessions=(5,)):
    records = []
    for session in sessions:
        eval_dir = os.path.join(root_dir, f'Session{session}', 'dialog', 'EmoEvaluation')
        wav_dir  = os.path.join(root_dir, f'Session{session}', 'sentences', 'wav')
        if not os.path.exists(eval_dir):
            print(f'Warning: Session{session} not found, skipping')
            continue
        for fname in sorted(os.listdir(eval_dir)):
            if not fname.endswith('.txt'):
                continue
            current_utt, current_emo, evaluator_votes = None, None, []
            with open(os.path.join(eval_dir, fname)) as f:
                for line in f:
                    stripped = line.strip()
                    if stripped.startswith('['):
                        if current_utt and current_emo in MAJORITY_MAP:
                            dialog   = '_'.join(current_utt.split('_')[:-1])
                            wav_path = os.path.join(wav_dir, dialog, current_utt + '.wav')
                            if os.path.exists(wav_path):
                                records.append({
                                    'utt_id': current_utt, 'wav_path': wav_path,
                                    'majority_emotion': MAJORITY_MAP[current_emo],
                                    'annotator_votes': list(evaluator_votes), 'session': session
                                })
                        parts = stripped.split('\t')
                        current_utt     = parts[1].strip() if len(parts) >= 3 else None
                        current_emo     = parts[2].strip() if len(parts) >= 3 else None
                        evaluator_votes = []
                        continue
                    if stripped.startswith('C-'):
                        parts = line.split('\t')
                        if len(parts) >= 2:
                            raw_vote = parts[1].strip().rstrip(';').lower()
                            mapped   = VOTE_MAP.get(raw_vote)
                            if mapped:
                                evaluator_votes.append(mapped)
                if current_utt and current_emo in MAJORITY_MAP:
                    dialog   = '_'.join(current_utt.split('_')[:-1])
                    wav_path = os.path.join(wav_dir, dialog, current_utt + '.wav')
                    if os.path.exists(wav_path):
                        records.append({
                            'utt_id': current_utt, 'wav_path': wav_path,
                            'majority_emotion': MAJORITY_MAP[current_emo],
                            'annotator_votes': list(evaluator_votes), 'session': session
                        })
    df = pd.DataFrame(records)
    print(f'Loaded {len(df)} utterances with annotator data')
    return df


def annotator_entropy(votes, base=2):
    if len(votes) < 2:
        return 0.0
    counts = np.array(list(Counter(votes).values()), dtype=float)
    counts /= counts.sum()
    return float(scipy_entropy(counts, base=base))


# ── Load & compute entropy ────────────────────────────────────────────────
print('Loading Session 5 annotator votes...')
df_votes = load_iemocap_annotator_votes(IEMOCAP_ROOT, sessions=(5,))
df_votes = df_votes[df_votes['annotator_votes'].apply(len) >= 2].copy()
df_votes['disagree_entropy'] = df_votes['annotator_votes'].apply(annotator_entropy)
print(f'{len(df_votes)} utterances with ≥2 categorical votes retained')
print(df_votes['disagree_entropy'].describe().round(4))


# ── Inference: KL score per annotated utterance ────────────────────────────
class IEMOCAPDatasetWithID(Dataset):
    def __init__(self, df, processor):
        self.df        = df.reset_index(drop=True)
        self.processor = processor
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            wav, sr = torchaudio.load(row['wav_path'])
            if sr != SAMPLE_RATE:
                wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)
            wav = wav.mean(dim=0)[:MAX_LENGTH]
            inp = self.processor(
                wav.numpy(), sampling_rate=SAMPLE_RATE,
                return_tensors='pt', padding='max_length',
                max_length=MAX_LENGTH, truncation=True
            )
            return {'input_values': inp['input_values'].squeeze(0),
                    'majority_label': torch.tensor(label2id[row['majority_emotion']], dtype=torch.long),
                    'valid': torch.tensor(1)}
        except Exception:
            return {'input_values': torch.zeros(MAX_LENGTH),
                    'majority_label': torch.tensor(0, dtype=torch.long),
                    'valid': torch.tensor(0)}

print('\nRunning inference on annotated utterances...')
load_checkpoint(full_model, 'stage3_best_fixed')
full_model.eval()

annotated_ds     = IEMOCAPDatasetWithID(df_votes, processor)
annotated_loader = DataLoader(annotated_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

kl_scores, explicit_preds, hidden_preds, majority_labels, valid_utt_ids = [], [], [], [], []
utt_id_list = df_votes['utt_id'].tolist()

with torch.no_grad():
    for batch_idx, batch in enumerate(annotated_loader):
        valid_mask = batch['valid'].bool()
        if not valid_mask.any():
            continue
        batch_start    = batch_idx * BATCH_SIZE
        global_indices = [batch_start + i for i in range(len(valid_mask)) if valid_mask[i]]
        valid_utt_ids.extend([utt_id_list[gi] for gi in global_indices if gi < len(utt_id_list)])

        iv     = batch['input_values'][valid_mask].to(DEVICE)
        labels = batch['majority_label'][valid_mask].to(DEVICE)

        hidden_enc             = full_model.encoder(iv)
        local_rep, global_rep, lw, gw = full_model.dual_attn(hidden_enc)
        exp_logits = full_model.explicit_head(global_rep)
        hid_logits = full_model.hidden_head(local_rep)

        eps     = 1e-8
        lw_norm = (lw + eps) / (lw + eps).sum(dim=-1, keepdim=True)
        gw_norm = (gw + eps) / (gw + eps).sum(dim=-1, keepdim=True)
        kl      = (lw_norm * (lw_norm / gw_norm).log()).sum(dim=-1)

        kl_scores.extend(kl.cpu().numpy())
        explicit_preds.extend(exp_logits.argmax(-1).cpu().numpy())
        hidden_preds.extend(hid_logits.argmax(-1).cpu().numpy())
        majority_labels.extend(labels.cpu().numpy())

kl_arr  = np.array(kl_scores)
exp_arr = np.array(explicit_preds)
hid_arr = np.array(hidden_preds)
maj_arr = np.array(majority_labels)

votes_indexed = df_votes.set_index('utt_id')
ent_arr = np.array([
    votes_indexed.loc[uid, 'disagree_entropy'] if uid in votes_indexed.index else np.nan
    for uid in valid_utt_ids
])
valid_ent_mask = ~np.isnan(ent_arr)
kl_arr  = kl_arr[valid_ent_mask]
exp_arr = exp_arr[valid_ent_mask]
hid_arr = hid_arr[valid_ent_mask]
maj_arr = maj_arr[valid_ent_mask]
ent_arr = ent_arr[valid_ent_mask]
print(f'Inference complete: {len(kl_arr)} utterances')


# ══════════════════════════════════════════════════════════
# CHECK A — DETECTION VALIDITY
# ══════════════════════════════════════════════════════════
print('\n' + '='*60)
print('CHECK A — DETECTION VALIDITY')
print('='*60)

ent_threshold = 0.0
gt_hidden     = (ent_arr > ent_threshold).astype(int)
if gt_hidden.mean() < 0.15:
    nonzero_ents  = ent_arr[ent_arr > 0]
    ent_threshold = np.percentile(nonzero_ents, 25)
    gt_hidden     = (ent_arr > ent_threshold).astype(int)
    print(f'  Using 25th percentile of nonzero entropies: {ent_threshold:.4f}')

print(f'  Entropy threshold : {ent_threshold:.4f}')
print(f'  Positive (ambiguous): {gt_hidden.sum()}/{len(gt_hidden)} ({100*gt_hidden.mean():.1f}%)')

kl_thresholds = {
    'mean+1std': kl_arr.mean() + kl_arr.std(),
    '75th pct':  np.percentile(kl_arr, 75),
    '90th pct':  np.percentile(kl_arr, 90),
}
for name, thresh in kl_thresholds.items():
    pred = (kl_arr > thresh).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(gt_hidden, pred, average='binary', zero_division=0)
    print(f'  KL>{name} ({thresh:.4f}): Flagged={pred.sum()} P={p:.3f} R={r:.3f} F1={f1:.3f}')

kl_mean_std = kl_arr.mean() + kl_arr.std()
try:
    auc = roc_auc_score(gt_hidden, kl_arr)
    print(f'\n  AUC-ROC: {auc:.4f}', end='  ')
    print('✓ Above-chance' if auc > 0.60 else ('~ Weak signal' if auc > 0.50 else '✗ Near-chance'))
except Exception as e:
    auc = 0.0; print(f'  AUC-ROC error: {e}')

stat, pval = mannwhitneyu(kl_arr[gt_hidden==1], kl_arr[gt_hidden==0], alternative='greater')
print(f'  Mann-Whitney: U={stat:.0f}, p={pval:.4f} | '
      f'Mean KL high={kl_arr[gt_hidden==1].mean():.4f} low={kl_arr[gt_hidden==0].mean():.4f}')
print(f'  {"✓ Significant" if pval < 0.05 else "✗ Not significant"} at p<0.05')


# ══════════════════════════════════════════════════════════
# CHECK B — HEAD DISAGREEMENT
# ══════════════════════════════════════════════════════════
print('\n' + '='*60)
print('CHECK B — EXPLICIT vs HIDDEN HEAD DISAGREEMENT')
print('='*60)

high_mask     = gt_hidden == 1
low_mask      = gt_hidden == 0
head_disagree = (exp_arr != hid_arr)

disagree_high = head_disagree[high_mask].mean()
disagree_low  = head_disagree[low_mask].mean()
print(f'  Head disagree on HIGH entropy: {disagree_high:.3f}')
print(f'  Head disagree on LOW  entropy: {disagree_low:.3f}')
print(f'  {"✓ Heads disagree more on ambiguous" if disagree_high > disagree_low else "✗ No differential disagreement"}')

smoking_gun = high_mask & head_disagree
n_sg        = smoking_gun.sum()
print(f'\n  Smoking gun utterances: {n_sg}')
if n_sg > 0:
    patterns = Counter(zip(
        [id2label[e] for e in exp_arr[smoking_gun]],
        [id2label[h] for h in hid_arr[smoking_gun]]
    ))
    print('  Top suppression patterns (expressed -> hidden):')
    for (exp_e, hid_e), count in patterns.most_common(6):
        print(f'    {exp_e:12s} -> {hid_e:12s}: {count:4d} ({100*count/n_sg:.1f}%)')


# ══════════════════════════════════════════════════════════
# CHECK C — DISTRIBUTION SHIFT
# ══════════════════════════════════════════════════════════
print('\n' + '='*60)
print('CHECK C — HIDDEN HEAD DISTRIBUTION SHIFT')
print('='*60)
for name, mask in [('HIGH entropy (ambiguous)', high_mask), ('LOW entropy (clear)', low_mask)]:
    counts = Counter(id2label[h] for h in hid_arr[mask])
    total  = mask.sum()
    print(f'\n  {name} (n={total}):')
    for emotion in EMOTIONS:
        c   = counts.get(emotion, 0)
        bar = '█' * int(30 * c / total)
        print(f'    {emotion:12s}: {c:4d} ({100*c/total:5.1f}%)  {bar}')


# ══════════════════════════════════════════════════════════
# VISUALISATION
# ══════════════════════════════════════════════════════════
fig = plt.figure(figsize=(15, 4))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

ax1 = fig.add_subplot(gs[0])
ax1.hist(kl_arr[low_mask],  bins=25, alpha=0.65, color='steelblue', label='Low entropy', density=True)
ax1.hist(kl_arr[high_mask], bins=25, alpha=0.65, color='tomato', label='High entropy', density=True)
ax1.axvline(kl_mean_std, color='black', ls='--', lw=1.2, label='KL threshold')
ax1.set_xlabel('KL Score'); ax1.set_ylabel('Density')
ax1.set_title('A: KL Score by Annotator Agreement'); ax1.legend(fontsize=7)

ax2 = fig.add_subplot(gs[1])
bars = ax2.bar(['Low entropy\n(clear)', 'High entropy\n(ambiguous)'],
               [disagree_low, disagree_high], color=['steelblue', 'tomato'], width=0.5, alpha=0.85)
ax2.set_ylabel('Head Disagreement Rate')
ax2.set_title('B: Explicit vs Hidden Head\nDisagreement Rate'); ax2.set_ylim(0, 1)
for bar, val in zip(bars, [disagree_low, disagree_high]):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.3f}',
             ha='center', fontsize=11, fontweight='bold')

ax3 = fig.add_subplot(gs[2])
x = np.arange(len(EMOTIONS)); w = 0.35
cnt_low  = [Counter(id2label[h] for h in hid_arr[low_mask]).get(e, 0) / low_mask.sum() for e in EMOTIONS]
cnt_high = [Counter(id2label[h] for h in hid_arr[high_mask]).get(e, 0) / high_mask.sum() for e in EMOTIONS]
ax3.bar(x - w/2, cnt_low,  width=w, color='steelblue', label='Low entropy',  alpha=0.85)
ax3.bar(x + w/2, cnt_high, width=w, color='tomato',    label='High entropy', alpha=0.85)
ax3.set_xticks(x); ax3.set_xticklabels(EMOTIONS, fontsize=8, rotation=15)
ax3.set_ylabel('Proportion'); ax3.set_title('C: Hidden Head Emotion\nDistribution Shift'); ax3.legend(fontsize=7)

plt.suptitle(
    f'Hidden Emotion Validation (FIXED): KL Score vs Annotator Disagreement\n'
    f'IEMOCAP Session 5 | N={len(kl_arr)} utterances | AUC-ROC={auc:.4f}',
    fontsize=11, fontweight='bold', y=1.02
)
plt.savefig('/content/drive/MyDrive/hidden_emotion_validation_fixed.png', dpi=150, bbox_inches='tight')
plt.savefig('/content/hidden_emotion_validation_fixed.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n' + '='*60)
print('PAPER-READY SUMMARY')
print('='*60)
print(f'  AUC-ROC          : {auc:.4f}')
print(f'  Mann-Whitney p   : {pval:.4f}')
print(f'  Head disagree    : {disagree_high:.3f} (ambiguous) vs {disagree_low:.3f} (clear)')
print(f'  Smoking gun cases: {n_sg}')